# End-to-end Ball-DP demo for Paper 2: nonconvex DP-SGD transcripts on MNIST embeddings

This notebook is the Paper 2 analogue of the ridge/ERM demo from Paper 1. It is not the official experiment runner; the official scripts added for Paper 2 are:

- `quantbayes/ball_dp/experiments/run_nonconvex_transcript_experiment.py`
- `quantbayes/ball_dp/experiments/run_nonconvex_transcript_attack_experiment.py`
- `quantbayes/ball_dp/experiments/aggregate_nonconvex_transcript.py`
- `quantbayes/ball_dp/experiments/aggregate_nonconvex_attack.py`

The notebook does one complete small run:

1. Load MNIST embeddings.
2. Build the theorem-backed one-hidden-layer tanh classifier with dense **operator-norm** constraints.
3. Choose a local Ball radius \(r\) from within-label embedding distances.
4. Train a Ball-DP Poisson DP-SGD release and a standard clipping-only comparator.
5. Compute per-step sensitivity summaries and finite-prior transcript ReRo bounds using the Hayes/global revealed-inclusion bound.
6. Run a finite-prior transcript MAP attack with known-inclusion and unknown-inclusion threat models.


## Mathematical setup

A record is \(z=(x,y)\), where \(x\in\mathbb R^d\). The local label-preserving metric is

$$
d((x,y),(x',y'))=
\begin{cases}
\|x-x'\|_2, & y=y',\\
\infty, & y\ne y'.
\end{cases}
$$

Paper 2 studies the **full noisy DP-SGD transcript**. With Poisson sampling and clipping threshold $C_t$, a local replacement within radius \(r\) has per-step sensitivity bounded by

$$
\Delta_t(r) = \min\{L_t r,\,2C_t\}.
$$

For the operator-norm tanh network used here, the source code gives a certified global Lipschitz constant $L_z$ from public theorem-side bounds. The training path uses this certificate to populate the Ball-DP and standard-DP ledgers.

For exact identification over a finite prior of size \(m\), the oblivious success probability is \(\kappa=1/m\). The main Paper 2 transcript bound uses

```python
ball_rero(release, prior=finite_prior, eta_grid=(0.5,), mode="hayes")
```

which evaluates the global revealed-inclusion product bound for the Poisson transcript. RDP is included below only as a familiar comparator; it is not the main transcript argument.


In [ ]:
# Notebook configuration

from __future__ import annotations

import dataclasses as dc
import math
import sys
from pathlib import Path

import jax
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "quantbayes").exists() and str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quantbayes.ball_dp.api import (
    attack_nonconvex_ball_trace_finite_prior,
    ball_rero,
    calibrate_ball_sgd_noise_multiplier,
    get_release_step_table,
    make_finite_identification_prior,
    make_trace_metadata_from_release,
)
from quantbayes.ball_dp.attacks.ball_policy import BallTraceMapAttackConfig
from quantbayes.ball_dp.attacks.gradient_based import DPSGDTraceRecorder, subtract_known_batch_gradients
from quantbayes.ball_dp.experiments.load_mnist_embeddings import load_or_create_mnist_resnet18_embeddings
from quantbayes.ball_dp.experiments.run_attack_experiment import (
    LoadedDataset,
    find_feasible_support_banks,
    make_support_set,
    resolve_dataset,
    select_ball_radius,
    summarize_embedding_ball_radii,
)
from quantbayes.ball_dp.theorem.registry import certified_lz, make_model
from quantbayes.ball_dp.theorem.specs import TheoremBounds, TheoremModelSpec, TrainConfig
from quantbayes.ball_dp.theorem.workflows import fit_release as theorem_fit_release
from quantbayes.ball_dp.types import ArrayDataset, Record

plt.rcParams.update({
    "figure.figsize": (7.0, 4.2),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "--",
    "legend.frameon": False,
})

# Data / cache knobs.
DATA_ROOT = "./data"
EMBEDDING_CACHE_PATH = None
FORCE_RECOMPUTE_EMBEDDINGS = False
ALLOW_CPU_FALLBACK = True
EMBEDDING_BATCH_SIZE = 256
NUM_WORKERS = 2

# Demo knobs. Use the official scripts below for full-data runs.
MAX_TRAIN_EXAMPLES = 5000
MAX_TEST_EXAMPLES = 2000
SUBSAMPLE_SEED = 0

# Paper-2-aligned model/accounting knobs.
HIDDEN_DIM = 128
A_BOUND = 4.0
LAMBDA_BOUND = 4.0
INPUT_BOUND_SCALE = 1.01
RADIUS_QUANTILE = 0.80
EPSILON = 8.0
DELTA = 1e-6
M_FINITE_PRIOR = 8
NUM_STEPS = 200
BATCH_SIZE = 128
CLIP_NORM = 1.0
LEARNING_RATE = 1e-3
ORDERS = tuple(list(range(2, 65)) + [80, 96, 128, 160, 192, 256])
RELEASE_SEED = 0


## Load MNIST embeddings

The loader caches ResNet18 embeddings. The first run can take time if the cache is absent; later runs should be faster. The demo optionally uses a class-balanced subsample to keep the notebook interactive. Omit the subsampling in official runs.


In [ ]:
X_train_j, y_train_j, X_test_j, y_test_j = load_or_create_mnist_resnet18_embeddings(
    data_root=DATA_ROOT,
    batch_size=EMBEDDING_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    require_jax_gpu=not ALLOW_CPU_FALLBACK,
    cache_path=EMBEDDING_CACHE_PATH,
    force_recompute=FORCE_RECOMPUTE_EMBEDDINGS,
)

X_train_full = np.asarray(jax.device_get(X_train_j), dtype=np.float32)
y_train_full = np.asarray(jax.device_get(y_train_j), dtype=np.int32).reshape(-1)
X_test_full = np.asarray(jax.device_get(X_test_j), dtype=np.float32)
y_test_full = np.asarray(jax.device_get(y_test_j), dtype=np.int32).reshape(-1)


def class_balanced_subsample(X, y, max_examples, *, seed):
    if max_examples is None or max_examples >= len(X):
        idx = np.arange(len(X), dtype=np.int64)
        return X, y, idx
    rng = np.random.default_rng(seed)
    labels = np.unique(y)
    per_class = max(1, int(max_examples) // max(1, len(labels)))
    chunks = []
    for label in labels:
        pool = np.flatnonzero(y == label)
        take = min(len(pool), per_class)
        chunks.append(rng.choice(pool, size=take, replace=False))
    idx = np.concatenate(chunks)
    if len(idx) < max_examples:
        rest = np.setdiff1d(np.arange(len(X)), idx, assume_unique=False)
        extra = rng.choice(rest, size=min(max_examples - len(idx), len(rest)), replace=False)
        idx = np.concatenate([idx, extra])
    rng.shuffle(idx)
    return X[idx], y[idx], idx

X_train, y_train, train_idx = class_balanced_subsample(
    X_train_full, y_train_full, MAX_TRAIN_EXAMPLES, seed=SUBSAMPLE_SEED
)
X_test, y_test, test_idx = class_balanced_subsample(
    X_test_full, y_test_full, MAX_TEST_EXAMPLES, seed=SUBSAMPLE_SEED + 17
)

num_classes = int(len(np.unique(y_train)))
feature_dim = int(X_train.reshape(len(X_train), -1).shape[1])

pd.DataFrame([
    {"split": "train", "n": len(X_train), "feature_dim": feature_dim, "num_classes": num_classes},
    {"split": "test", "n": len(X_test), "feature_dim": feature_dim, "num_classes": int(len(np.unique(y_test)))},
])


## Build the dense operator-norm theorem model

The model specification below is the one to use for Paper 2 experiments:

```python
TheoremModelSpec(
    d_in=feature_dim,
    hidden_dim=128,
    task="multiclass",
    parameterization="dense",
    constraint="op",
    num_classes=num_classes,
)
```

The public input bound \(B\) is set to a small margin above the empirical train/test maximum embedding norm for this demo. In a paper run, treat \(B\), \(A\), and \(\Lambda\) as public pre-declared theorem constants.


In [ ]:
X_all = np.concatenate([X_train.reshape(len(X_train), -1), X_test.reshape(len(X_test), -1)], axis=0)
B_all = float(np.max(np.linalg.norm(X_all, axis=1)) * INPUT_BOUND_SCALE)

task = "binary" if num_classes == 2 else "multiclass"
spec = TheoremModelSpec(
    d_in=feature_dim,
    hidden_dim=HIDDEN_DIM,
    task=task,
    parameterization="dense",
    constraint="op",
    num_classes=None if task == "binary" else num_classes,
)
bounds = TheoremBounds(B=B_all, A=A_BOUND, Lambda=LAMBDA_BOUND)
L_z = certified_lz(spec, bounds)

# Explicitly instantiate the model exactly as in the Paper 2 source path.
_demo_model = make_model(spec, key=jr.PRNGKey(0), init_project=True, bounds=bounds)

pd.DataFrame([
    {
        "task": task,
        "d_in": feature_dim,
        "hidden_dim": HIDDEN_DIM,
        "constraint": "op",
        "B": B_all,
        "A": A_BOUND,
        "Lambda": LAMBDA_BOUND,
        "certified_L_z": L_z,
        "critical_radius_2C_over_Lz": 2.0 * CLIP_NORM / L_z,
    }
])


## Choose the Ball radius

The demo uses the same style of local radius as Paper 1: the maximum over labels of the within-label empirical \(q80\) pairwise embedding distance,

$$
r=\max_c \widehat Q_{0.80}\{\|x_i-x_j\|_2:y_i=y_j=c\}.
$$

The per-step Ball sensitivity is clipped by \(2C\), so the radius is informative only when \(L_z r < 2C\) for at least some steps. The table below reports the selected radius and the critical radius $2C/L_z$.


In [ ]:
radius_report = summarize_embedding_ball_radii(
    X_train,
    y_train,
    quantiles=(0.50, 0.80, 0.95),
    max_exact_pairs=250_000,
    max_sampled_pairs=100_000,
    seed=0,
)
r_ball = float(select_ball_radius(
    radius_report,
    strategy="max_labelwise_quantile",
    quantile=RADIUS_QUANTILE,
    allow_observed_max=False,
))

radius_df = pd.DataFrame([
    {"quantity": "selected local Ball radius", "value": r_ball},
    {"quantity": "critical radius 2C/L_z", "value": 2.0 * CLIP_NORM / L_z},
    {"quantity": "raw L_z r", "value": L_z * r_ball},
    {"quantity": "standard clipping sensitivity 2C", "value": 2.0 * CLIP_NORM},
    {"quantity": "raw Ball/standard ratio", "value": min(L_z * r_ball, 2.0 * CLIP_NORM) / (2.0 * CLIP_NORM)},
])
radius_df


In [ ]:
fig, ax = plt.subplots()
ax.bar(radius_df["quantity"], radius_df["value"])
ax.set_title("Radius and sensitivity scale diagnostics")
ax.set_ylabel("value")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
plt.show()


## Calibrate noise and train transcript releases

Both releases use Poisson sampling and the same network/training hyperparameters. The Ball release calibrates against the Ball-DP ledger; the standard comparator calibrates against the standard clipping ledger.


In [ ]:
def calibrate_noise(mechanism: str):
    privacy = "ball_dp" if mechanism == "ball" else "standard_dp"
    return calibrate_ball_sgd_noise_multiplier(
        dataset_size=len(X_train),
        radius=r_ball,
        lz=L_z,
        num_steps=NUM_STEPS,
        batch_size=BATCH_SIZE,
        clip_norm=CLIP_NORM,
        target_epsilon=EPSILON,
        delta=DELTA,
        privacy=privacy,
        batch_sampler="poisson",
        accountant_subsampling="match_sampler",
        orders=ORDERS,
        lower=1e-3,
        upper=0.25,
        max_upper=256.0,
        num_bisection_steps=14,
    )

ball_cal = calibrate_noise("ball")
std_cal = calibrate_noise("standard")

pd.DataFrame([
    {"mechanism": "ball", **{k: v for k, v in ball_cal.items() if k in {"epsilon", "delta", "noise_multiplier", "alpha"}}},
    {"mechanism": "standard", **{k: v for k, v in std_cal.items() if k in {"epsilon", "delta", "noise_multiplier", "alpha"}}},
])


In [ ]:
def train_release(mechanism: str, *, noise_multiplier: float, seed: int, trace_recorder=None):
    privacy = "ball_dp" if mechanism == "ball" else "standard_dp"
    init_key, train_key = jr.split(jr.PRNGKey(seed), 2)
    model = make_model(spec, key=init_key, init_project=True, bounds=bounds)
    train_cfg = TrainConfig(
        radius=r_ball,
        privacy=privacy,
        epsilon=EPSILON,
        delta=DELTA,
        num_steps=NUM_STEPS,
        batch_size=BATCH_SIZE,
        batch_sampler="poisson",
        accountant_subsampling="match_sampler",
        clip_norm=CLIP_NORM,
        noise_multiplier=float(noise_multiplier),
        learning_rate=LEARNING_RATE,
        checkpoint_selection="last",
        eval_every=50,
        eval_batch_size=1024,
        normalize_noisy_sum_by="batch_size",
        seed=seed,
    )
    return theorem_fit_release(
        model,
        spec,
        bounds,
        X_train,
        y_train,
        train_cfg=train_cfg,
        X_eval=X_test,
        y_eval=y_test,
        key=train_key,
        trace_recorder=trace_recorder,
        orders=ORDERS,
        record_operator_norms=True,
        operator_norms_every=50,
        warn_if_ball_equals_standard=False,
    )

ball_release = train_release("ball", noise_multiplier=float(ball_cal["noise_multiplier"]), seed=RELEASE_SEED)
std_release = train_release("standard", noise_multiplier=float(std_cal["noise_multiplier"]), seed=RELEASE_SEED)

release_df = pd.DataFrame([
    {
        "mechanism": "ball",
        "accuracy": ball_release.utility_metrics.get("accuracy", np.nan),
        "public_eval_loss": ball_release.utility_metrics.get("public_eval_loss", np.nan),
        "noise_multiplier": float(ball_cal["noise_multiplier"]),
        "epsilon_ball_view": ball_release.privacy.ball.dp_certificates[0].epsilon,
        "epsilon_standard_view": ball_release.privacy.standard.dp_certificates[0].epsilon,
        "release_kind": ball_release.release_kind,
    },
    {
        "mechanism": "standard",
        "accuracy": std_release.utility_metrics.get("accuracy", np.nan),
        "public_eval_loss": std_release.utility_metrics.get("public_eval_loss", np.nan),
        "noise_multiplier": float(std_cal["noise_multiplier"]),
        "epsilon_ball_view": std_release.privacy.ball.dp_certificates[0].epsilon,
        "epsilon_standard_view": std_release.privacy.standard.dp_certificates[0].epsilon,
        "release_kind": std_release.release_kind,
    },
])
release_df


In [ ]:
fig, ax = plt.subplots()
ax.bar(release_df["mechanism"], release_df["accuracy"])
ax.set_ylim(0, 1)
ax.set_ylabel("public test accuracy")
ax.set_title("Utility of one transcript release")
plt.show()

fig, ax = plt.subplots()
ax.bar(release_df["mechanism"], release_df["noise_multiplier"])
ax.set_ylabel("noise multiplier")
ax.set_title("Calibrated noise multiplier")
plt.show()


## Inspect per-step sensitivities

The public theorem certificate gives a uniform \(L_z\), so with fixed \(C\) and fixed \(r\) the raw Ball sensitivity is the same at each step in this demo:

$$
\Delta_t^{\mathrm{Ball}}=\min\{L_z r,2C\},\qquad
\Delta_t^{\mathrm{Std}}=2C.
$$

The transcript ledger stores these quantities per step because the code also supports schedules.


In [ ]:
step_table = pd.DataFrame(get_release_step_table(ball_release))
display(step_table.head())

step_summary = pd.DataFrame([
    {
        "mean_sample_rate": step_table["sample_rate"].mean(),
        "mean_delta_ball": step_table["delta_ball"].mean(),
        "mean_delta_standard": step_table["delta_standard"].mean(),
        "mean_ball_to_standard_ratio": step_table["ball_to_standard_ratio"].mean(),
        "max_direct_c_ball": step_table["direct_c_ball"].max(),
        "max_direct_c_standard": step_table["direct_c_standard"].max(),
    }
])
step_summary


In [ ]:
fig, ax = plt.subplots()
ax.plot(step_table["step"], step_table["delta_ball"], label="Ball sensitivity")
ax.plot(step_table["step"], step_table["delta_standard"], label="standard sensitivity")
ax.set_xlabel("SGD step")
ax.set_ylabel(r"$\Delta_t$")
ax.set_title("Per-step transcript sensitivity")
ax.legend()
plt.show()


## Finite-prior transcript ReRo bounds

For exact identification over a uniform finite support \(S\) with \(|S|=m\), use the finite-exact prior helper. The sample locations do not affect \(\kappa=1/m\); they are included only to satisfy the common prior interface.


In [ ]:
finite_prior = make_finite_identification_prior(
    np.zeros((M_FINITE_PRIOR, feature_dim), dtype=np.float32),
    weights=None,
)


def report_row(name: str, release, primary: str):
    hayes = ball_rero(release, prior=finite_prior, eta_grid=(0.5,), mode="hayes")
    rdp = ball_rero(release, prior=finite_prior, eta_grid=(0.5,), mode="rdp")
    hp = hayes.points[0]
    rp = rdp.points[0]
    return {
        "mechanism": name,
        "m": M_FINITE_PRIOR,
        "kappa": 1.0 / M_FINITE_PRIOR,
        "primary_bound_hayes": hp.gamma_ball if primary == "ball" else hp.gamma_standard,
        "bound_hayes_ball_view": hp.gamma_ball,
        "bound_hayes_standard_view": hp.gamma_standard,
        "primary_bound_rdp": rp.gamma_ball if primary == "ball" else rp.gamma_standard,
        "bound_rdp_ball_view": rp.gamma_ball,
        "bound_rdp_standard_view": rp.gamma_standard,
        "hayes_mode": hayes.mode,
    }

bound_df = pd.DataFrame([
    report_row("ball", ball_release, "ball"),
    report_row("standard", std_release, "standard"),
])
bound_df


In [ ]:
fig, ax = plt.subplots()
ax.bar(bound_df["mechanism"], bound_df["primary_bound_hayes"])
ax.axhline(1.0 / M_FINITE_PRIOR, linestyle="--", label="oblivious 1/m")
ax.set_ylim(0, 1)
ax.set_ylabel(r"$B_{1/m}^{test}$")
ax.set_title("Hayes/global transcript ReRo bound")
ax.legend()
plt.show()


## One finite-prior transcript attack

The attack below constructs a finite candidate support \(S\) around one training anchor. It then replaces that anchor with a uniformly chosen support member and records the full DP-SGD transcript.

Two Bayes/MAP variants are evaluated:

- **Known inclusion:** the attacker knows which retained Poisson steps included the target index.
- **Unknown inclusion:** the attacker sees only the noisy residual transcript and marginalizes over Bernoulli inclusion.

The residualization step subtracts the known contribution of non-target batch members. This is an intentionally strong oracle attack; it is useful because it tests whether the theorem-backed transcript bound is in the right qualitative regime.


In [ ]:
dataset_spec = resolve_dataset("MNIST-embeddings")
data_for_supports = LoadedDataset(
    spec=dataset_spec,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    label_values=np.unique(np.concatenate([y_train, y_test])),
    num_classes=num_classes,
    feature_dim=feature_dim,
    empirical_embedding_bound=float(np.max(np.linalg.norm(X_train, axis=1))),
    backend=jax.default_backend(),
)

banks = find_feasible_support_banks(
    data_for_supports,
    radius_value=r_ball,
    max_required_m=M_FINITE_PRIOR,
    num_supports=1,
    anchor_seed=0,
    source_mode="public_only",
    max_search=3000,
    strict=True,
    anchor_selection="rare_class",
)
bank = banks[0]
X_support, y_support, source_ids, support_dists = make_support_set(
    bank,
    m=M_FINITE_PRIOR,
    draw_index=0,
    base_seed=0,
    selection="farthest",
)

target_position = 0
true_x = np.asarray(X_support[target_position], dtype=np.float32)
true_y = int(y_support[target_position])

X_trial = np.asarray(X_train, dtype=np.float32).copy()
y_trial = np.asarray(y_train, dtype=np.int32).copy()
X_trial[int(bank.anchor_index)] = true_x
y_trial[int(bank.anchor_index)] = true_y

pd.DataFrame([
    {
        "anchor_index": int(bank.anchor_index),
        "anchor_label": int(bank.anchor_label),
        "m": M_FINITE_PRIOR,
        "target_position": target_position,
        "target_source_id": source_ids[target_position],
        "support_max_distance_to_anchor": float(np.max(support_dists)),
        "radius": r_ball,
    }
])


In [ ]:
# Retrain on the replacement dataset X_trial and retain the full noisy transcript.
# This is intentionally separate from the utility release above.
def train_attack_trial():
    init_key, train_key = jr.split(jr.PRNGKey(RELEASE_SEED + 123), 2)
    model = make_model(spec, key=init_key, init_project=True, bounds=bounds)
    train_cfg = TrainConfig(
        radius=r_ball,
        privacy="ball_dp",
        epsilon=EPSILON,
        delta=DELTA,
        num_steps=NUM_STEPS,
        batch_size=BATCH_SIZE,
        batch_sampler="poisson",
        accountant_subsampling="match_sampler",
        clip_norm=CLIP_NORM,
        noise_multiplier=float(ball_cal["noise_multiplier"]),
        learning_rate=LEARNING_RATE,
        checkpoint_selection="last",
        eval_every=50,
        eval_batch_size=1024,
        normalize_noisy_sum_by="batch_size",
        seed=RELEASE_SEED + 123,
    )
    return theorem_fit_release(
        model,
        spec,
        bounds,
        X_trial,
        y_trial,
        train_cfg=train_cfg,
        X_eval=X_test,
        y_eval=y_test,
        key=train_key,
        trace_recorder=recorder,
        orders=ORDERS,
        record_operator_norms=False,
        warn_if_ball_equals_standard=False,
    )

recorder = DPSGDTraceRecorder(capture_every=1, keep_models=True, keep_batch_indices=True)
attack_release = train_attack_trial()

trace_meta = make_trace_metadata_from_release(
    attack_release,
    target_index=int(bank.anchor_index),
    extra={
        "anchor_index": int(bank.anchor_index),
        "anchor_label": int(bank.anchor_label),
        "support_selection": "farthest",
        "target_position": int(target_position),
    },
)
trace = recorder.to_trace(
    state=attack_release.extra.get("model_state", None),
    loss_name=spec.loss_name,
    reduction=str(trace_meta.get("reduction", "mean")),
    metadata=trace_meta,
)
residual_trace = subtract_known_batch_gradients(
    trace,
    ArrayDataset(X_trial, y_trial, name="trial_train"),
    target_index=int(bank.anchor_index),
    loss_name=spec.loss_name,
    seed=0,
)

pd.DataFrame([
    {
        "recorded_steps": len(trace.steps),
        "sample_rate_mean": np.mean(trace_meta.get("sample_rates", [])),
        "attack_release_accuracy": attack_release.utility_metrics.get("accuracy", np.nan),
        "residualized": residual_trace.metadata.get("residualized_against_known_batch", False),
    }
])


In [ ]:
true_record = Record(features=true_x, label=true_y)
attack_rows = []
for mode in ["known_inclusion", "unknown_inclusion"]:
    cfg = BallTraceMapAttackConfig(
        mode=mode,
        step_mode="all" if mode == "known_inclusion" else "all",
        seed=0,
    )
    attack = attack_nonconvex_ball_trace_finite_prior(
        residual_trace,
        X_support,
        y_support,
        cfg=cfg,
        target_index=int(bank.anchor_index),
        known_label=int(bank.anchor_label),
        true_record=true_record,
        eta_grid=(0.5,),
    )
    row = {"mode": mode, "status": attack.status}
    row.update(attack.metrics)
    row.update({
        "predicted_prior_index": attack.diagnostics.get("predicted_prior_index"),
        "true_prior_index": attack.diagnostics.get("true_prior_index"),
        "selected_step_count": attack.diagnostics.get("selected_step_count"),
    })
    attack_rows.append(row)

attack_df = pd.DataFrame(attack_rows)
attack_df


## Reading the demo outputs

The notebook is a sanity check, not a theorem proof and not a final experimental table. The quantities to compare against Paper 2 are:

- `mean_ball_to_standard_ratio`: whether the local transcript sensitivity is strictly below standard clipping.
- `primary_bound_hayes`: the finite-prior exact-identification bound \(B_{1/m}^{test}\) from the Hayes/global transcript calculation.
- `exact_identification_success` / `prior_rank`: empirical finite-prior attack behavior in one strong oracle trial.
- `accuracy`: public utility of the final released model.

The official scripts repeat these measurements across datasets, seeds, epsilons, support draws, and anchors.
